In [1]:
import json
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from torch.utils.data import Dataset
import os

# 1. Load and prepare dataset
with open('../data/chatbot_intents_full.json') as f:
    data = json.load(f)

# Create training samples
samples = []
for intent in data['intents']:
    for pattern in intent['patterns']:
        samples.append({
            'text': pattern,
            'label': intent['tag']
        })

df = pd.DataFrame(samples)
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])

/opt/miniconda3/envs/recommender/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2. Dataset Class
class ChatDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
            'labels': torch.tensor(self.labels[idx])
        }
        return item

    def __len__(self):
        return len(self.labels)

In [3]:
# 3. Tokenization and Data Split
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Split data
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# Tokenize with smaller max_length
train_encodings = tokenizer(
    train_df['text'].tolist(), 
    truncation=True, 
    padding=True,
    max_length=128
)
val_encodings = tokenizer(
    val_df['text'].tolist(), 
    truncation=True, 
    padding=True,
    max_length=128
)

train_dataset = ChatDataset(train_encodings, train_df['label'].tolist())
val_dataset = ChatDataset(val_encodings, val_df['label'].tolist())

In [4]:
# 4. Model Setup
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_encoder.classes_)
)

Loading weights: 100%|█| 100/100 [00:00<00:00, 1506.11it/s, Materializing param=
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# 5. Training Configuration
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=20,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=False   # 🚨 IMPORTANT
)


In [6]:
# 6. Metrics
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    acc = (preds == p.label_ids).mean()
    return {"accuracy": acc}

In [7]:
# 7. Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics   # optional, if defined
)

In [8]:
# 8. Train
trainer.train()

# Save model and tokenizer
save_path = "./chatbot_model"
model.save_pretrained(save_path, safe_serialization=False)
tokenizer.save_pretrained(save_path)

# Save label encoder
import joblib
joblib.dump(label_encoder, os.path.join(save_path, 'label_encoder.joblib'))

/opt/miniconda3/envs/recommender/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.665353,0.121212
2,No log,2.610025,0.212121
3,No log,2.489022,0.303030
4,No log,2.354252,0.424242
5,No log,2.201180,0.515152
6,No log,2.076565,0.575758
7,No log,1.929327,0.727273
8,No log,1.809815,0.727273
9,No log,1.684184,0.727273
10,No log,1.568901,0.727273


Writing model shards: 100%|███████████████████████| 1/1 [00:00<00:00,  4.68it/s]
/opt/miniconda3/envs/recommender/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|███████████████████████| 1/1 [00:00<00:00,  3.90it/s]
/opt/miniconda3/envs/recommender/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|███████████████████████| 1/1 [00:00<00:00,  2.52it/s]
/opt/miniconda3/envs/recommender/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|███████

['./chatbot_model/label_encoder.joblib']

In [15]:
def predict_response(text):
    text_lower = text.lower().strip()
    
    # Rule-based shortcuts - Return ONLY the string
    if text_lower in ["hi", "hello", "hey", "good morning", "good evening"]:
        return "Hello! How can I help you today?"
    if text_lower in ["thanks", "thank you", "thx", "appreciate it"]:
        return "You're welcome! 😊"

    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1)
            confidence, predicted_idx = torch.max(probs, dim=1)
            
        predicted_tag = label_encoder.inverse_transform([predicted_idx.item()])[0]

        # Confidence-based fallback - Return ONLY the string
        if confidence.item() < 0.6:
            return "I want to make sure I understand. Are you asking about an order, a return, or something else?"

        # Normal response - Return ONLY the string (removed parentheses and comma)
        return response_map.get(predicted_tag, "I'm not sure how to help with that.")

    except Exception as e:
        print(f"Prediction error: {str(e)}")
        return "Sorry, something went wrong."

In [16]:
# Create interface
interface = gr.Interface(
    fn=predict_response,
    inputs=gr.Textbox(lines=2, placeholder="Ask me anything..."),
    outputs="text",
    title="Customer Support Chatbot",
    examples=[
        ["How do I reset my password?"],
        ["Where is my order?"],
        ["What's your return policy?"]
    ]
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
